In [1]:
from pathlib import Path
import glob
import csv
import os

# -----------------------------
# PATHS (EDIT if needed)
# -----------------------------
BASE_PATH = Path(r"C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance")
ON_TIME_PATH = BASE_PATH / "data" / "01_OnTime_Performance"
OUTPUT_PATH = ON_TIME_PATH / "merged"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

YEARS = [2023, 2024, 2025]

# -----------------------------
# HELPERS
# -----------------------------
def is_lfs_pointer(csv_path: Path) -> bool:
    """Detect Git LFS pointer files (not real CSV data)."""
    with open(csv_path, "r", encoding="utf-8", errors="replace") as f:
        first_line = f.readline().strip()
    return first_line.startswith("version https://git-lfs.github.com/spec/v1")


def merge_year_raw(year: int) -> Path | None:
    """
    RAW merge monthly/part files into one big file (binary concat).
    Assumes all parts have identical header.
    """
    year_dir = ON_TIME_PATH / str(year)
    files = sorted(glob.glob(str(year_dir / f"OnTime_{year}_*.csv")))
    files = [Path(f) for f in files]

    if not files:
        print(f"❌ No files found for {year}: {year_dir}")
        return None

    # fail fast if LFS pointer
    bad = [f for f in files if is_lfs_pointer(f)]
    if bad:
        print(f"\n❌ Found Git LFS pointer files for {year} (not real CSV data):")
        for f in bad[:10]:
            print(" -", f)
        print("Fix: run in repo: git lfs install && git lfs pull && git lfs checkout")
        return None

    output_file = OUTPUT_PATH / f"OnTime_{year}_ALL.csv"
    if output_file.exists():
        output_file.unlink()  # prevent accidental append

    print(f"\n📦 RAW merging {len(files)} files for {year} -> {output_file.name}")
    for f in files:
        print(" -", f.name)

    with open(output_file, "wb") as out:
        for i, f in enumerate(files):
            with open(f, "rb") as inp:
                if i == 0:
                    out.write(inp.read())      # keep header
                else:
                    inp.readline()             # skip header
                    out.write(inp.read())

    print(f"✅ Created: {output_file}")
    return output_file


def combine_year_files(year_files: list[Path], out_file: Path) -> Path:
    """
    Combine already-merged year files into one (binary concat).
    """
    if out_file.exists():
        out_file.unlink()

    print(f"\n📊 Combining years -> {out_file.name}")
    with open(out_file, "wb") as out:
        for i, f in enumerate(year_files):
            print(" - adding:", f.name)
            with open(f, "rb") as inp:
                if i == 0:
                    out.write(inp.read())      # keep header
                else:
                    inp.readline()             # skip header
                    out.write(inp.read())

    print(f"🎉 Created: {out_file}")
    return out_file


# -----------------------------
# 2024 FIX (43 columns -> 42 columns)
# -----------------------------
def fix_2024_extra_column(input_csv: Path, output_csv: Path) -> dict:
    """
    Fix rows in 2024 that have 43 fields instead of expected 42.
    Strategy:
      - Read header to get expected column count.
      - For any row with 43 fields:
          - Remove ONE extra blank field if present.
          - Otherwise drop the extra last field.
      - Keep good 42-field rows unchanged.
    Writes a corrected CSV to output_csv.

    Returns stats dict.
    """
    stats = {"total_rows": 0, "kept_rows": 0, "fixed_rows": 0, "dropped_rows": 0}

    # Safety: never write into the same file we're reading
    if input_csv.resolve() == output_csv.resolve():
        raise ValueError("Input and output CSV cannot be the same path. Use a temp output then replace.")

    print(f"\n🧩 Fixing 2024 file: {input_csv.name}")
    print(f"➡️ Output: {output_csv.name}")

    with open(input_csv, "r", encoding="utf-8", errors="replace", newline="") as fin:
        reader = csv.reader(fin)

        header = next(reader)
        expected = len(header)

        with open(output_csv, "w", encoding="utf-8", newline="") as fout:
            writer = csv.writer(fout)
            writer.writerow(header)

            for row in reader:
                stats["total_rows"] += 1
                n = len(row)

                if n == expected:
                    writer.writerow(row)
                    stats["kept_rows"] += 1
                    continue

                if n == expected + 1:
                    # common: an extra empty field inserted somewhere
                    if "" in row:
                        row.remove("")  # remove first empty
                    else:
                        row = row[:-1]  # drop last field

                    if len(row) == expected:
                        writer.writerow(row)
                        stats["fixed_rows"] += 1
                    else:
                        stats["dropped_rows"] += 1
                    continue

                # If row is very broken, drop it
                stats["dropped_rows"] += 1

    print("✅ 2024 FIX COMPLETE")
    print("   total_rows  :", stats["total_rows"])
    print("   kept_rows   :", stats["kept_rows"])
    print("   fixed_rows  :", stats["fixed_rows"])
    print("   dropped_rows:", stats["dropped_rows"])

    return stats


# ============================================================
# RUN PIPELINE
# ============================================================

# 1) RAW merge each year from parts
merged_2023 = merge_year_raw(2023)
merged_2024 = merge_year_raw(2024)
merged_2025 = merge_year_raw(2025)

if not (merged_2023 and merged_2024 and merged_2025):
    raise SystemExit("\n❌ Stop: Missing year merges. Fix above issues first.")

# 2) FIX 2024 (write temp then overwrite safely)
temp_fixed_2024 = OUTPUT_PATH / "OnTime_2024_ALL_FIXED.csv"
fix_2024_extra_column(merged_2024, temp_fixed_2024)

# Overwrite original safely
temp_fixed_2024.replace(merged_2024)
print(f"✅ Overwrote original: {merged_2024.name}")

# 3) Combine 2023 + FIXED 2024 + 2025
combined = OUTPUT_PATH / "OnTime_2023_2025_ALL.csv"
combine_year_files([merged_2023, merged_2024, merged_2025], combined)

print("\nDONE ✅")
print("Merged outputs are located in:", OUTPUT_PATH)



📦 RAW merging 12 files for 2023 -> OnTime_2023_ALL.csv
 - OnTime_2023_1.csv
 - OnTime_2023_10.csv
 - OnTime_2023_11.csv
 - OnTime_2023_12.csv
 - OnTime_2023_2.csv
 - OnTime_2023_3.csv
 - OnTime_2023_4.csv
 - OnTime_2023_5.csv
 - OnTime_2023_6.csv
 - OnTime_2023_7.csv
 - OnTime_2023_8.csv
 - OnTime_2023_9.csv
✅ Created: C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance\data\01_OnTime_Performance\merged\OnTime_2023_ALL.csv

📦 RAW merging 12 files for 2024 -> OnTime_2024_ALL.csv
 - OnTime_2024_1.csv
 - OnTime_2024_10.csv
 - OnTime_2024_11.csv
 - OnTime_2024_12.csv
 - OnTime_2024_2.csv
 - OnTime_2024_3.csv
 - OnTime_2024_4.csv
 - OnTime_2024_5.csv
 - OnTime_2024_6.csv
 - OnTime_2024_7.csv
 - OnTime_2024_8.csv
 - OnTime_2024_9.csv
✅ Created: C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance\data\01_OnTime_Performance\merged\OnTime_2024_ALL.csv

📦 RAW merging 10 files for 2025 -> OnTime_2025_ALL.csv
 - OnTime_2025_1.csv
 - OnTime_202

In [3]:
import pandas as pd

p = r"C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance\data\01_OnTime_Performance\merged\OnTime_2023_2025_ALL.csv"
df = pd.read_csv(p, low_memory=False)
print(df.shape)
print(sorted(df["YEAR"].unique()))   
print(df.groupby(["YEAR","MONTH"]).size().head(20))


(19775745, 42)
[np.int64(2023), np.int64(2024), np.int64(2025)]
YEAR  MONTH
2023  1        538837
      2        502749
      3        580322
      4        561441
      5        579958
      6        577262
      7        601866
      8        602987
      9        569338
      10       598968
      11       563777
      12       570394
2024  1        547271
      2        519221
      3        591767
      4        582205
      5        609743
      6        611132
      7        634613
      8        619025
dtype: int64


In [4]:
from pathlib import Path

BASE_PATH = Path(r"C:\Users\salsi\Desktop\uni\420\FinalProject\us-airline-operational-performance")
MERGED_PATH = BASE_PATH / "data" / "01_OnTime_Performance" / "merged"

files = [
    MERGED_PATH / "OnTime_2023_ALL.csv",
    MERGED_PATH / "OnTime_2024_ALL.csv",
    MERGED_PATH / "OnTime_2025_ALL.csv",
    MERGED_PATH / "OnTime_2023_2025_ALL.csv",
]

for file in files:
    print("\n" + "="*60)
    print(f"📄 Header check: {file.name}")
    print("="*60)

    with open(file, 'r', encoding='utf-8', errors='replace') as f:
        header = f.readline().strip()

    columns = header.split(",")
    print(f"Column count: {len(columns)}")
    print(columns)


📄 Header check: OnTime_2023_ALL.csv
Column count: 42
['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_STATE_NM', 'DEST_AIRPORT_ID', 'DEST', 'DEST_CITY_NAME', 'DEST_STATE_ABR', 'DEST_STATE_NM', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'DEP_DELAY_NEW', 'DEP_DEL15', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'ARR_DELAY_NEW', 'ARR_DEL15', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

📄 Header check: OnTime_2024_ALL.csv
Column count: 42
['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM', 'OP_CARRIER_FL_NUM', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'ORIGIN_CITY_NAME', 'ORIGIN_STATE_ABR', 'ORIGIN_